In [1]:
#| default_exp noising_time

In [2]:
#|export
import numpy as np
import torch


In [3]:
#| export
def cosine_beta_scheduler(T, s=0.008):
    steps = T + 1
    t = np.linspace(0, T, steps) / T
    alpha_bar = np.cos((t + s) / (1 + s) * np.pi / 2) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return np.clip(betas, 1e-5, 0.999)

#| export
def timestep_embedding(timesteps, dim):
    half_dim = dim // 2
    emb = np.log(10000) / (half_dim - 1)
    emb = np.exp(np.arange(half_dim) * -emb)
    emb = timesteps[:, None] * emb[None, :]
    emb = np.concatenate([np.sin(emb), np.cos(emb)], axis=1)
    return torch.from_numpy(emb).float()

#| export
def noisify(T, x0, alphas_bar):
  device = x0.device
  bs = x0.shape[0]
  t = torch.randint(0, T, size=(bs,), device=device)
  eps = torch.randn_like(x0)
  a_bar = alphas_bar[t].unsqueeze(1)
  xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * eps
  return xt,t,eps
